In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import pandas as pd
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader, random_split

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
import copy
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train = MNIST(root='data/', train=True,  download=True, transform=transform)
test  = MNIST(root='data/', train=False, download=True, transform=transform)

train_data, holdout = random_split(
    train, [42000, 18000],
    generator=torch.Generator().manual_seed(SEED)
)
validation_data, test_data = random_split(
    holdout, [9000, 9000],
    generator=torch.Generator().manual_seed(SEED)
)



In [ ]:
TRAIN_LOADER = DataLoader(train_data, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
VALIDATION_LOADER   = DataLoader(validation_data,   batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
TEST_LOADER  = DataLoader(test_data,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)



In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.suptitle('MNIST Sample Images')

class_samples = {i: [] for i in range(10)}
for image, label in train:
    if len(class_samples[label]) < 2:
        class_samples[label].append(image)
    if all(len(v) == 2 for v in class_samples.values()):
        break

for col, digit in enumerate(range(10)):
    for row in range(2):
        axes[row, col].imshow(class_samples[digit][row].squeeze())
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f'Digit {digit}')

plt.tight_layout()
plt.savefig('samples.png')
plt.show()


In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32,3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(6272, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) 
        x = x.view(x.size(0), -1)             
        x = F.relu(self.fc1(x))               
        return self.fc2(x)                    


class DualBlockCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,  32,3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64,3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(3136, 256)
        self.bn3   = nn.BatchNorm1d(256)
        self.fc2   = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  
        x = x.view(x.size(0), -1)                       
        x = F.relu(self.bn3(self.fc1(x)))               
        return self.fc2(x)                             


class DeepRegularisedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1a = nn.Conv2d(1,  32, 3, padding=1)
        self.bn1a   = nn.BatchNorm2d(32)
        self.conv1b = nn.Conv2d(32, 32, 3, padding=1)
        self.bn1b   = nn.BatchNorm2d(32)
        self.conv2a = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2a   = nn.BatchNorm2d(64)
        self.conv2b = nn.Conv2d(64, 64, 3, padding=1)
        self.bn2b   = nn.BatchNorm2d(64)
        self.conv3  = nn.Conv2d(64, 128,3, padding=1)
        self.bn3    = nn.BatchNorm2d(128)
        self.pool    = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(1152, 256)
        self.bn4 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = self.pool(F.relu(self.bn1b(self.conv1b(x))))  
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = self.pool(F.relu(self.bn2b(self.conv2b(x))))  
        x = self.pool(F.relu(self.bn3(self.conv3(x))))    
        x = x.view(x.size(0), -1)                         
        x = self.dropout(x)
        x = F.relu(self.bn4(self.fc1(x)))                
        x = self.dropout(x)
        return self.fc2(x)                                


In [ ]:
def train_model(model, model_name):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3,
    )

    history = {
        'trainLoss': [], 'validationLoss': [],
        'trainAccuracy':  [], 'validationAccuracy':  []
    }

    bestvalidationLoss = float('inf')
    bestState = None
    count = 0

    for epoch in range(25):
        model.train()
        tallyLoss, correct, total = 0.0, 0, 0
        for images, labels in TRAIN_LOADER:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            tallyLoss += loss.item() * images.size(0)
            _, predictions = torch.max(outputs, 1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        trainLoss = tallyLoss / total
        trainAccuracy = correct / total

        model.eval()
        validationLoss, validation_correct, validationTotal = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in VALIDATION_LOADER:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                validationLoss += loss.item() * images.size(0)
                _, predictions = torch.max(outputs, 1)
                validation_correct += (predictions == labels).sum().item()
                validationTotal += labels.size(0)

        validationLoss /= validationTotal
        validationAccuracy = validation_correct / validationTotal

        history['trainLoss'].append(trainLoss)
        history['validationLoss'].append(validationLoss)
        history['trainAccuracy'].append(trainAccuracy)
        history['validationAccuracy'].append(validationAccuracy)

        scheduler.step(validationLoss)

        if validationLoss < bestvalidationLoss:
            bestvalidationLoss = validationLoss
            bestState = copy.deepcopy(model.state_dict())
            count = 0
        else:
            count += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'[{model_name}] Epoch {epoch+1:3d}/{25} | '
                  f'Train Loss: {trainLoss:.4f}  Acc: {trainAccuracy:.4f} | '
                  f'validation Loss: {validationLoss:.4f}  Acc: {validationAccuracy:.4f}')

        if count >= 7:
            print(f'[{model_name}] Early stopping at epoch {epoch+1}.')
            break

    print(f'[{model_name}] Training complete. '
          f'Best validation loss: {bestvalidationLoss:.4f}')

    model.load_state_dict(bestState)
    return history, model

In [ ]:
def Evalidationuate(model):
    model.eval()
    correct, total = 0, 0
    allPredictions, allLabels = [], []

    with torch.no_grad():
        for images, labels in TEST_LOADER:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predictions = torch.max(outputs, 1)
            correct += (predictions == labels).sum().item()
            total   += labels.size(0)
            allPredictions.extend(predictions.cpu().numpy())
            allLabels.extend(labels.cpu().numpy())

    accuracy = correct / total
    return accuracy, np.array(allPredictions), np.array(allLabels)


In [ ]:
ShallowCNN = ShallowCNN()
DualBlockCNN = DualBlockCNN()
DeepRegularisedCNN = DeepRegularisedCNN()



In [ ]:
historyShallowCNN, ShallowCNN = train_model(
    ShallowCNN,model_name='ShallowCNN'
)

accuracyShallowCNN, predictionsShallowCNN, labelsShallowCNN = Evalidationuate(ShallowCNN)
print(f'\nShallowCNN — Test Accuracy: {accuracyShallowCNN:.4f} ({accuracyShallowCNN*100:.2f}%)')


In [ ]:
historyDualBlockCNN, DualBlockCNN = train_model(
    DualBlockCNN,
    model_name='DualBlockCNN'
)

accuracyDualBlockCNN, predictionsDualBlockCNN, labelsDualBlockCNN = Evalidationuate(DualBlockCNN, TEST_LOADER)
print(f'\nDualBlockCNN — Test Accuracy: {accuracyDualBlockCNN:.4f} ({accuracyDualBlockCNN*100:.2f}%)')



In [ ]:
historyDeepRegularisedCNN, DeepRegularisedCNN = train_model(
    DeepRegularisedCNN,
    model_name='DeepRegularisedCNN'
)

accuaracyDeepRegularisedCNN, predictionsDeepRegularisedCNN, labelsDeepRegularisedCNN = Evalidationuate(DeepRegularisedCNN, TEST_LOADER)
print(f'\nDeepRegularisedCNN — Test Accuracy: {accuaracyDeepRegularisedCNN:.4f} ({accuaracyDeepRegularisedCNN*100:.2f}%)')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Learning Curves',)

DeepRegularisedCNNonfigs = [
    ('ShallowCNN',         historyShallowCNN, '#2196F3'),
    ('DualBlockCNN',       historyDualBlockCNN, '#4CAF50'),
    ('DeepRegularisedCNN', historyDeepRegularisedCNN, '#FF5722'),
]

for col, (name, hist, colour) in enumerate(DeepRegularisedCNNonfigs):
    epochs = range(0, len(hist['trainLoss']))

    axes[0, col].plot(epochs, hist['trainLoss'],
                      color=colour, label='Train',)
    axes[0, col].plot(epochs, hist['validationLoss'], color=colour,
                      ls='--', label='validation', )
    axes[0, col].set_title(f'{name}\nLoss', )
    axes[0, col].set_xlabel('Epoch')
    axes[0, col].set_ylabel('Cross-Entropy Loss')
    axes[0, col].legend()
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(epochs, [a*100 for a in hist['trainAccuracy']],
                      color=colour, label='Train', linewidth=2)
    axes[1, col].plot(epochs, [a*100 for a in hist['validationAccuracy']],
                      color=colour, )
    axes[1, col].set_title(f'{name}\nAccuracy')
    axes[1, col].set_xlabel('Epoch')
    axes[1, col].set_ylabel('Accuracy')
    axes[1, col].set_ylim([85, 100])
    axes[1, col].legend()
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png',)
plt.show()
print('Fig 2: Learning curves (loss and accuracy) for all three models.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Confusion Matrices on Test Set', )

model_results = [
    ('ShallowCNN',         predictionsShallowCNN, labelsShallowCNN),
    ('DualBlockCNN',       predictionsDualBlockCNN, labelsDualBlockCNN),
    ('DeepRegularisedCNN', predictionsDeepRegularisedCNN, labelsDeepRegularisedCNN),
]

for ax, (name, predictions, labels) in zip(axes, model_results):
    cm = confusion_matrix(labels, predictions)
    ConfusionMatrixDisplay(confusion_matrix=cm,
                           display_labels=list(range(10))).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.tight_layout()
plt.savefig('confusion_matrices.png')
plt.show()
print('Fig 3: Confusion matrices for all three models.')

In [ ]:
class_names = [str(i) for i in range(10)]

for name, predictions, labels in [
    ('ShallowCNN',         predictionsShallowCNN, labelsShallowCNN),
    ('DualBlockCNN',       predictionsDualBlockCNN, labelsDualBlockCNN),
    ('DeepRegularisedCNN', predictionsDeepRegularisedCNN, labelsDeepRegularisedCNN),
]:
    print(f'\n{"─"*55}')
    print(f'  {name} — Classification Report')
    print(f'{"─"*55}')
    print(classification_report(labels, predictions, target_names=class_names))


In [ ]:
summary = pd.DataFrame({
    'Model':            ['ShallowCNN', 'DualBlockCNN', 'DeepRegularisedCNN'],
    'Conv Blocks':      [1, 2, 3],
    'Batch Norm':       ['No', 'Yes', 'Yes'],
    'Dropout':          ['No', 'No', 'Yes (0.4)'],
    'Best validation Accuracy': [
        round(max(historyShallowCNN['validationAccuracy']) * 100, 2),
        round(max(historyDualBlockCNN['validationAccuracy']) * 100, 2),
        round(max(historyDeepRegularisedCNN['validationAccuracy']) * 100, 2),
    ],
    'Test Accuracy': [round(accuracyShallowCNN*100, 2), round(accuracyDualBlockCNN*100, 2), round(accuaracyDeepRegularisedCNN*100, 2)],
    'Epochs Trained': [
        len(historyShallowCNN['trainLoss']),
        len(historyDualBlockCNN['trainLoss']),
        len(historyDeepRegularisedCNN['trainLoss']),
    ]
})

print('\nSummary Comparison Table')
print(summary.to_string(index=False))
